# SchoolBridge — LayoutXLM 본격 학습 (자동 라벨링)

**목적**: 2,064 PDF 자동 라벨링 데이터로 LayoutXLM fine-tune. 본문/표 모두 단일 모델로 sentence boundary 결정 — 룰 의존 0.

**입력**: `layoutxlm_train.jsonl` (자동 라벨링, page 단위 record) + PDF zip
**출력**: `layoutxlm_best.pt`

## 업로드 파일
1. `layoutxlm_train.jsonl` — 라벨링 데이터 (pdf, page, words, bboxes, labels)
2. `pdfs.zip` — PDF 폴더 압축 (학습 시 page 이미지 동적 렌더링)

**Colab 세팅**: 런타임 → T4 GPU

## 1. 환경 + 설치

In [ ]:
!pip install -q transformers sentencepiece pdfplumber pymupdf pillow
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print("설치 완료")

In [ ]:
import torch
import detectron2
from transformers import LayoutXLMProcessor, LayoutLMv2ForTokenClassification
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("detectron2:", detectron2.__version__)

## 2. 학습 데이터 + PDF zip 업로드

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, os

uploaded = files.upload()
for name in uploaded:
    print(f'  {name}: {len(uploaded[name])/1024/1024:.1f} MB')

# PDF zip 압축 풀기
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('.')
        print(f'{name} extracted')

# PDF_DIR — 명시적 후보 시도 (rglob 자동 감지가 . 로 잘못 잡는 케이스 방지)
PDF_DIR = None
for cand in [Path('all_pdfs'), Path('pdfs') / 'all_pdfs', Path('pdfs'), Path('.')]:
    if cand.exists():
        pdfs_here = list(cand.glob('*.pdf'))
        if pdfs_here:
            PDF_DIR = cand
            break

assert PDF_DIR is not None, 'PDF_DIR not found — zip 풀린 위치 확인 필요'
n_pdfs = len(list(PDF_DIR.rglob('*.pdf')))
print(f'PDF_DIR: {PDF_DIR}')
print(f'PDF count: {n_pdfs}')

## 3. 학습 데이터 로드

In [ ]:
records = []
with open('layoutxlm_train.jsonl', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# 통계
n_pdfs_unique = len({r['pdf'] for r in records})
max_label = max(max(r['labels']) for r in records if r['labels'])
print(f'Page records: {len(records)}')
print(f'Unique PDFs: {n_pdfs_unique}')
print(f'Max group_id: {max_label}')
print(f'avg words per page: {sum(r["n_words"] for r in records)/len(records):.0f}')

# PDF 파일 누락 체크
missing = [r['pdf'] for r in records if not (PDF_DIR / r['pdf']).exists()]
missing_unique = set(missing)
print(f'Missing PDFs (in zip): {len(missing_unique)}')
if missing_unique:
    print('Sample missing:', list(missing_unique)[:3])
    records = [r for r in records if (PDF_DIR / r['pdf']).exists()]
    print(f'Kept records: {len(records)}')

## 4. LayoutXLM 모델 + Dataset

In [ ]:
from transformers import LayoutXLMProcessor, LayoutLMv2ForTokenClassification
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import fitz, torch, random, time

MODEL_ID = "microsoft/layoutxlm-base"

# MAX_SENT_ID: 한 PDF 안 sentence 최대 개수 + 1 (unlabeled용)
# 자동 라벨링에서 max_label 확인 후 여유 두고 설정
MAX_SENT_ID = max_label + 2  # unlabeled = max_label + 1
NUM_LABELS = MAX_SENT_ID + 1

processor = LayoutXLMProcessor.from_pretrained(MODEL_ID, apply_ocr=False)
model = LayoutLMv2ForTokenClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_LABELS,
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')
print(f'Device: {device}')
print(f'NUM_LABELS: {NUM_LABELS} (max_sent_id+2 for unlabeled+padding)')


class LayoutXLMDataset(Dataset):
    def __init__(self, records, pdf_dir, processor, num_labels):
        self.records = records
        self.pdf_dir = pdf_dir
        self.processor = processor
        self.unlabeled_id = num_labels - 1  # last label for unlabeled

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        pdf_path = self.pdf_dir / rec['pdf']
        # PDF page 동적 렌더링
        doc = fitz.open(pdf_path)
        page = doc[rec['page']]
        pix = page.get_pixmap(dpi=150)
        image = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
        doc.close()

        words = rec['words']
        boxes = rec['bboxes']
        word_labels = [(l if l >= 0 else self.unlabeled_id) for l in rec['labels']]

        encoded = self.processor(
            image, words, boxes=boxes,
            return_tensors='pt', truncation=True,
            padding='max_length', max_length=512,
        )
        word_ids = encoded.word_ids()
        token_labels = []
        for wid in word_ids:
            if wid is None:
                token_labels.append(-100)
            else:
                token_labels.append(word_labels[wid] if wid < len(word_labels) else -100)
        encoded['labels'] = torch.tensor(token_labels)
        return {k: v.squeeze(0) for k, v in encoded.items()}


# train/val split
random.seed(42)
random.shuffle(records)
n_val = max(50, len(records) // 10)
val_records = records[:n_val]
train_records = records[n_val:]

train_ds = LayoutXLMDataset(train_records, PDF_DIR, processor, NUM_LABELS)
val_ds = LayoutXLMDataset(val_records, PDF_DIR, processor, NUM_LABELS)

BATCH = 2  # LayoutXLM 360M, T4 16GB에서 안전한 크기
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
print(f'Train: {len(train_records)} | Val: {len(val_records)}')
print(f'Batches: train {len(train_loader)} | val {len(val_loader)}')

## 5. 학습

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
EPOCHS = 5
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val = float('inf')
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    n_batches = 0
    t0 = time.time()
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        n_batches += 1
        if n_batches % 50 == 0:
            print(f'  step {n_batches}/{len(train_loader)} loss={loss.item():.4f}')
    scheduler.step()

    model.eval()
    val_loss = 0.0
    correct = 0; total = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            val_loss += outputs.loss.item()
            preds = outputs.logits.argmax(-1)
            labels = batch['labels']
            mask = labels != -100
            correct += ((preds == labels) & mask).sum().item()
            total += mask.sum().item()

    train_loss /= max(n_batches, 1)
    val_loss /= max(len(val_loader), 1)
    acc = correct / max(total, 1)
    elapsed = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  acc={acc:.3f}  ({elapsed:.0f}s)')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            'state_dict': model.state_dict(),
            'model_id': MODEL_ID,
            'num_labels': NUM_LABELS,
            'max_sent_id': MAX_SENT_ID,
        }, 'layoutxlm_best.pt')
        print('  > saved best')

print(f'Best val: {best_val:.4f}')

## 6. best.pt 다운로드

In [ ]:
from google.colab import files
files.download('layoutxlm_best.pt')